https://docs.langchain.com/oss/python/langchain/rag

In [1]:
#!pip install langchain langchain-text-splitters langchain-community bs4
!pip install -U "langchain-openai"
!pip install -U "langchain-core"
#!pip install langchain==0.3.11
#!pip uninstall -y langchain langchain-core langchain-community
!pip install -U langchain
!pip install langchain langchain-community langchain-text-splitters langchain-chroma chromadb

In [2]:
import os
from langchain.chat_models import init_chat_model

import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

model = init_chat_model("gpt-5.2")

OPENAI_API_KEY: ··········


In [3]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [4]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [7]:
from langchain_community.document_loaders import TextLoader
#from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import chromadb
# Load the text file
loader = TextLoader("/content/testdreamjournal2.txt")
docs = loader.load()

    # Split into chunks using /end as separator
text_splitter = CharacterTextSplitter(
    separator="/end",
    chunk_overlap= 0, #200,
    keep_separator=False
    # chunk_size=2000
)
doc_chunks = text_splitter.split_documents(docs)

In [8]:
document_ids = vector_store.add_documents(documents=doc_chunks)

print(document_ids[:3])

['7394e845-79e7-4e6d-a5ec-67a913e75e59']


In [9]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [10]:
from langchain.agents import create_agent


tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a dream journal. "
    "Use the tool to help answer user queries."
)
agent = create_agent(model, tools, system_prompt=prompt)

In [11]:
query = (
    "is there a dream about animals?\n\n"

)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

is there a dream about animals?


================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_rC9I9cZWMvkVFhjNhOMf7ucc)
 Call ID: call_rC9I9cZWMvkVFhjNhOMf7ucc
  Args:
    query: dream about animals
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': '/content/testdreamjournal2.txt'}
Content: Dream 1: The Dog and the Bone
Date: 12/05/2015

I dream about a dog that gave me a bone.

/end

Dream 2: The Floating Library
Date: 03/18/2016

I was in a giant library where all the books floated around me. When I opened one, I was pulled into the story itself. In one book, I was soaring through the clouds on the back of a golden eagle, feeling the wind rush past me.

/end

Dream 3: Reverse City
Date: 09/02/2016

I walked through a city where everything was upside down. The roads were abo

In [12]:
query = (
    "can you return the full dream that has the golden eagle?\n\n"

)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

can you return the full dream that has the golden eagle?


================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_AsWM7DY5X6FKo5GdjFFGtkTy)
 Call ID: call_AsWM7DY5X6FKo5GdjFFGtkTy
  Args:
    query: full dream golden eagle
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': '/content/testdreamjournal2.txt'}
Content: Dream 1: The Dog and the Bone
Date: 12/05/2015

I dream about a dog that gave me a bone.

/end

Dream 2: The Floating Library
Date: 03/18/2016

I was in a giant library where all the books floated around me. When I opened one, I was pulled into the story itself. In one book, I was soaring through the clouds on the back of a golden eagle, feeling the wind rush past me.

/end

Dream 3: Reverse City
Date: 09/02/2016

I walked through a city where everything was up